In [27]:
import numpy as np
import pandas as pd
from collections import Counter
import warnings, os, gc, re
import matplotlib.pyplot as plt
import psutil
import swifter

from ast import literal_eval

%matplotlib inline

warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.4f}'.format

In [254]:
df_px_info = pd.read_excel('01. Patient Information.xlsx')
df_admission = pd.read_excel('02. Admission.xlsx')
df_treatment = pd.read_excel('03. Treatment received.xlsx')
df_daily_PE = pd.read_excel('04. Daily Physical Exam.xlsx')
df_lab_test = pd.read_excel('05. Laboratory tests.xlsx')
df_quant = pd.read_excel('06. RT-PCR SARS CoV Quantitative.xlsx')
df_chest_day1 = pd.read_excel('07. Chest Imaging - Day 1.xlsx')
df_chest_improv = pd.read_excel('07. Chest Imaging - improved or worse.xlsx')
df_ekg = pd.read_excel('08. EKG Main Findings.xlsx')
df_end_trial = pd.read_excel('09. End of Trial.xlsx')
df_concomitant = pd.read_excel('10. Concomitant Medications.xlsx')
df_adverse = pd.read_excel('11. Adverse Events.xlsx')

In [3]:
df_adverse.head()

,Study Num,Description,Date of onset,Dose of the study drug,Severity,Seriousness,Treatment,Action taken with the study drug,Outcome,Causal relationship with the study drug,Comment
0,4,loose bowel movement,2020-11-06,NaN,Mild,No,"Yes, Erceflora 1 vial orally TID",Dose NOT changed,Improved and for Discharge,Not related,NaN
1,5,Patient had fall and had contusion hematoma,2020-11-07,NaN,Mild,No,"Yes, Chlorhexidine oral care",Dose NOT changed,No improvement,Not related,Patient accidentally slipped on the bathroom f...
2,5,Patient had bouts of loose stools (5x),2020-11-08,NaN,Mild,No,"Yes, Racecadptril",Dose NOT changed,Improved and for Discharge,Not related,No diarrhea at Day 4
3,8,severe epigastric pain,2020-11-24,4.0000,Moderate,No,No,Dose NOT changed,No improvement,Not related,epigastric pain can be related to the COVID-19...
4,8,severe epigastric pain 9/10,2020-11-27,5.0000,Moderate,No,No,Drug Withdrawn,Improved and for Discharge,Related,epigastric pain can be multifactorial: 1. poss...


### Merge ALL rows (Failed Attempt; DO NOT RUN)

In [25]:
df1 = df_px_info.merge(df_admission, how='outer', on='Study Num')
df1.head() 

,Study Num,Hospital Number,Last Name,First Name,Middle Name,Suffix,Sex,Birthdate,Date of Enrollment,Site,...,Severe Liver Impairment? (ALT>10x),"Has received other drugs CQ, HCQ, Lpv/Rit?",Severe Renal Impairment needing HD?,History of Gout or hyperuricemia?,Mental status change?,Has receiver favipiravir?,Pregnant?,Unable to consent to use condom for study +7 days,Female unable to consent contraception during study +7 days?,Male whose partner unable to consent contraception for study +7days?
0,1,4768849,A,M,R,NaN,Male,1979-03-31,2020-10-17,UP - PHILIPPINE GENERAL HOSPITAL,...,N,N,N,N,N,N,N,N,N,N
1,2,39509,R,A,D,JR,Male,1977-08-11,2020-10-22,DR. JOSE N. RODRIGUEZ MEMORIAL HOSPITAL,...,N,N,N,N,N,N,N,N,N,N
2,3,3579613,J,K,T,NOTAP,Female,1978-07-05,2020-10-31,UP - PHILIPPINE GENERAL HOSPITAL,...,N,N,N,N,N,N,N,N,N,N
3,4,1271950,O,W,A,NOTAP,Male,1959-10-01,2020-11-04,QUIRINO MEMORIAL MEDICAL CENTER,...,N,N,N,N,N,N,N,N,N,N
4,5,1271953,O,Y,G,NaN,Female,1953-02-17,2020-11-06,QUIRINO MEMORIAL MEDICAL CENTER,...,N,N,N,N,N,N,N,N,N,N


In [21]:
df1.columns

Index(['Study Num', 'Hospital Number', 'Last Name', 'First Name',
       'Middle Name', 'Suffix', 'Sex', 'Birthdate', 'Date of Enrollment',
       'Site', 'Study Arm', 'Date Admitted', 'Date of 1st COVID symptom',
       'Fever', 'Onset', 'Cough', 'Onset.1', 'Shortness of Breath', 'Onset.2',
       'Body Malaise', 'Onset.3', 'Chills', 'Onset.4', 'Headache', 'Onset.5',
       'Loss of Smell', 'Onset.6', 'Loss of Taste', 'Onset.7', 'Diarrhea',
       'Onset.8', 'Fatigue', 'Onset.9', 'Sore throat', 'Onset.10',
       'Other Symptoms', 'Onset.11', 'Specific other symptom', 'Hypertension',
       'Asthma', 'Lung conditions', 'Cardiac conditions', 'Previous stroke',
       'Cancers', 'Immunocompromised state', 'Gout/Hyperuricemia', 'SBP',
       'DBP', 'HR', 'RR', 'Temp', 'Pulse Ox at Room Air', 'Date Done',
       'Infiltrates', 'Infiltrates (specify)', 'Pneumonia',
       'Pulmonary Congestion', 'Pleura Effusion', 'Other findings',
       'Baseline Positive COVID test Date', 'Result', 'Pre

In [80]:
df1 = df_px_info.merge(df_admission, how='inner', on='Study Num').drop_duplicates()
df1 = df1.merge(df_treatment, how='right', on='Study Num').drop_duplicates()
df1 = df1.merge(df_daily_PE, how='right', on='Study Num').drop_duplicates()
df1 = df1.merge(df_lab_test, how='right', on='Study Num').drop_duplicates()
# df1 = df1.merge(df_quant, how='left', on='Study Num').drop_duplicates()
# df1 = df1.merge(df_chest, how='left', on='Study Num').drop_duplicates()
# df1 = df1.merge(df_ekg, how='left', on='Study Num').drop_duplicates()
# df1 = df1.merge(df_end_trial, how='left', on='Study Num').drop_duplicates()
# df1 = df1.merge(df_concomitant, how='left', on='Study Num').drop_duplicates()
# df1 = df1.merge(df_adverse, how='left', on='Study Num').drop_duplicates()
df1.head()

,Study Num,Hospital Number,Last Name,First Name,Middle Name,Suffix,Sex,Birthdate,Date of Enrollment,Site,...,Na,Cl,FBS,CRP,Procal,Ferritin,DDimer,Urine RBC,Urine Sugar,Urine Protein
0,1,4768849,A,M,R,NaN,Male,1979-03-31,2020-10-17,UP - PHILIPPINE GENERAL HOSPITAL,...,137,99,90,192,0.1300,348,5.5700,260,0,0
1,1,4768849,A,M,R,NaN,Male,1979-03-31,2020-10-17,UP - PHILIPPINE GENERAL HOSPITAL,...,137,99,90,192,0.1300,348,5.5700,260,0,0
2,1,4768849,A,M,R,NaN,Male,1979-03-31,2020-10-17,UP - PHILIPPINE GENERAL HOSPITAL,...,137,99,90,192,0.1300,348,5.5700,260,0,0
3,1,4768849,A,M,R,NaN,Male,1979-03-31,2020-10-17,UP - PHILIPPINE GENERAL HOSPITAL,...,137,99,90,192,0.1300,348,5.5700,260,0,0
4,1,4768849,A,M,R,NaN,Male,1979-03-31,2020-10-17,UP - PHILIPPINE GENERAL HOSPITAL,...,137,99,90,192,0.1300,348,5.5700,260,0,0


In [81]:
len(df1)

332050

In [82]:
df1.to_csv('merged_rows.csv')

In [84]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 332050 entries, 0 to 333245
Columns: 159 entries, Study Num to Urine Protein
dtypes: datetime64[ns](19), float64(15), int64(1), object(124)
memory usage: 405.3+ MB


### Date of Lysis of Fever

In [11]:
# day 0 Temp is the Admission excel's Temp field, with Time set to 'AM Time' by default for grouping later
df_day_0 = df_admission.loc[df_admission['Fever'] == 'Y'][['Study Num', 'Temp']]
df_day_0['Day'] = 'Day 0'
df_day_0['Time'] = 'AM Time'
df_day_0.head(10)

,Study Num,Temp,Day,Time
0,1,37.8000,Day 0,AM Time
1,2,36.8000,Day 0,AM Time
2,3,39.4000,Day 0,AM Time
3,4,38.5000,Day 0,AM Time
4,5,37.7000,Day 0,AM Time
6,8,36.7000,Day 0,AM Time
7,9,36.7000,Day 0,AM Time
8,10,36.6000,Day 0,AM Time
9,11,37.7000,Day 0,AM Time
10,12,36.5000,Day 0,AM Time


In [93]:
df_day_x = df_daily_PE.merge(df_admission, how='left', on='Study Num') \
                    .drop_duplicates()[['Study Num', 'Temp_x', 'Day', 'Time', 'Fever']]
df_day_x.columns = ['Study Num', 'Temp', 'Day', 'Time', 'Fever']
df_day_x = df_day_x[(df_day_x['Fever'] == 'Y') & (df_day_x['Temp'].notna()) & (df_day_x['Temp'] <= 42) & \
                    (df_day_x['Temp'] > 34) & (df_day_x['Day'].str.contains('Day'))]
df_day_x.drop('Fever', axis=1, inplace=True)
len(df_day_x)

1981

In [94]:
df_day_x.head(20)

,Study Num,Temp,Day,Time
0,1,36.5000,Day 1,AM Time
1,1,36.6000,Day 1,PM Time
2,1,36.5000,Day 2,AM Time
3,1,36.8000,Day 2,PM Time
4,1,36.5000,Day 3,AM Time
5,1,36.0000,Day 3,PM Time
6,1,36.6000,Day 4,AM Time
7,1,36.5000,Day 4,PM Time
8,1,36.6000,Day 5,AM Time
9,1,36.5000,Day 5,PM Time


In [95]:
# merge rows from day 0 (Admission.xls) and day x (Daily Physical Exam.xls)
frames = [df_day_0, df_day_x]
df_lysis = pd.concat(frames).reset_index(drop=True)
df_lysis['Day'] = df_lysis['Day'].str.strip('Day ').astype('int')
df_lysis = df_lysis.sort_values(by=['Study Num', 'Day', 'Time']) 
df_lysis.head(20)    

,Study Num,Temp,Day,Time
0,1,37.8000,0,AM Time
100,1,36.5000,1,AM Time
101,1,36.6000,1,PM Time
102,1,36.5000,2,AM Time
103,1,36.8000,2,PM Time
104,1,36.5000,3,AM Time
105,1,36.0000,3,PM Time
106,1,36.6000,4,AM Time
107,1,36.5000,4,PM Time
108,1,36.6000,5,AM Time


In [96]:
df_lysis_max = df_lysis.groupby(['Study Num', 'Day'])['Temp'].agg(['max'])
df_lysis_max['Fever?'] = np.where(df_lysis_max['max'] > 37.5, 'Yes', 'No')
df_lysis_max.head(30)

max Fever?
Study Num Day               
1         0   37.8000    Yes
          1   36.6000     No
          2   36.8000     No
          3   36.5000     No
          4   36.6000     No
          5   36.6000     No
          6   36.4000     No
          7   36.9000     No
          8   36.5000     No
          9   36.3000     No
          10  36.8000     No
          11  36.7000     No
          12  36.4000     No
          13  36.7000     No
          14  36.1000     No
2         0   36.8000     No
          1   36.9000     No
          2   36.5000     No
          3   36.9000     No
          4   36.2000     No
          5   36.5000     No
          6   36.6000     No
          7   36.8000     No
          8   36.7000     No
          9   36.2000     No
          10  36.6000     No
          11  36.7000     No
          12  36.8000     No
          13  36.8000     No
          28  36.5000     No

In [98]:
df_lysis_max.to_excel('lysis.xlsx')

### Mean earliest time of CXR Improvement

In [176]:
df_chest_img = pd.read_excel('07. Chest Imaging - improved or worse.xlsx')
df_chest_img.head()

,Study Num,Day,Date of Test,Improved,Worse,Describe
0,1,Day 4,2020-10-20,N,Y,GROUND GLASS OPACITIES ARE NOW NOTED IN BOTH L...
1,1,Day 7,2020-10-23,Y,N,IMPRESSION: â€¢ PNEUMONIA WITH SIGNS OF REGRES...
2,1,Day 10,2020-10-27,Y,N,PNEUMONIA WITH SIGNS OF REGRESSION. Linear opa...
3,2,Day 4,2020-10-25,Y,N,Minimal clearing of haziness in the right uppe...
4,2,Day 7,2020-10-28,N,N,"No significant chane in the opacities, noted a..."


In [180]:
df_chest_img_1 = df_chest_img[df_chest_img['Improved'] == 'Y'].sort_values(by=['Study Num', 'Day']) \
                    .drop(['Worse', 'Describe'], axis=1)
df_chest_img_1['Day'] = df_chest_img['Day'].str.strip('Day ').astype('int')
df_chest_img_1.head()

,Study Num,Day,Date of Test,Improved
2,1,10,2020-10-27,Y
1,1,7,2020-10-23,Y
6,2,13,2020-11-03,Y
7,2,28,2020-11-18,Y
3,2,4,2020-10-25,Y


In [182]:
df_min_day = df_chest_img_1.groupby(['Study Num'])['Day'].agg(['min']).reset_index() \
                        .set_axis(['Study Num', 'Day'], axis=1, inplace=False)
df_min_day.head(10)

,Study Num,Day
0,1,7
1,2,4
2,3,4
3,4,4
4,5,4
5,6,7
6,8,4
7,9,7
8,12,4
9,13,4


In [183]:
len(df_min_day)

92

In [189]:
df_chest_min = df_min_day.merge(df_chest_img_1, how='left', on=['Study Num', 'Day']).drop_duplicates()
df_chest_min.head()

,Study Num,Day,Date of Test,Improved
0,1,7,2020-10-23,Y
1,2,4,2020-10-25,Y
2,3,4,2020-11-03,Y
3,4,4,2020-11-09,Y
4,5,4,2020-11-09,Y


In [190]:
len(df_chest_min)

92

In [192]:
# df_chest_min.to_excel('test.xlsx')

In [196]:
df_cx_improv = df_chest_min.merge(df_admission[['Study Num','Date Admitted']], \
                                  how='left', on=['Study Num']).drop_duplicates().drop(['Improved'], axis=1)
df_cx_improv.head()

,Study Num,Day,Date of Test,Date Admitted
0,1,7,2020-10-23,2020-10-09
1,2,4,2020-10-25,2020-10-21
2,3,4,2020-11-03,2020-10-29
3,4,4,2020-11-09,2020-11-03
4,5,4,2020-11-09,2020-11-05


In [198]:
df_cx_improv['Time of CXR Improvement'] = (df_cx_improv['Date of Test'] - df_cx_improv['Date Admitted']).dt.days
df_cx_improv.head()

,Study Num,Day,Date of Test,Date Admitted,Time of CXR Improvement
0,1,7,2020-10-23,2020-10-09,14
1,2,4,2020-10-25,2020-10-21,4
2,3,4,2020-11-03,2020-10-29,5
3,4,4,2020-11-09,2020-11-03,6
4,5,4,2020-11-09,2020-11-05,4


In [260]:
df_cx_with_study = df_cx_improv.merge(df_px_info[['Study Num','Study Arm']], \
                                  how='left', on=['Study Num'])
df_cx_with_study.head()

,Study Num,Day,Date of Test,Date Admitted,Time of CXR Improvement,Study Arm
0,1,7,2020-10-23,2020-10-09,14,Control
1,2,4,2020-10-25,2020-10-21,4,Control
2,3,4,2020-11-03,2020-10-29,5,Control
3,4,4,2020-11-09,2020-11-03,6,Favipiravir
4,5,4,2020-11-09,2020-11-05,4,Favipiravir


In [261]:
df_cx_with_study.to_excel('slide 24 - earliest time CXR.xlsx', index=False)

### Merging Attempt (3 Huge Tables)
Run from top of every merge peg

In [204]:
## 1st merge
## Key: Study_Num, Day, Time
## merging guides: https://pandas.pydata.org/pandas-docs/stable/user_guide/merging.html
df_merge_1 = df_daily_PE.merge(df_treatment, how='outer', on=['Study Num', 'Day', 'Time']).drop_duplicates()
len(df_merge_1)

3355

In [205]:
df_merge_1.head()

,Study Num,Day,Time,SBP,DBP,HR,RR,Temp,Pulse Ox,Score Px Condition,...,Score Pleural Effusion,Score Thoracic Rales,Score Mental Status,NEWS Score,STATUS Score,Any ADVERSE EVENT,Supportive Care,Favipiravir 9tabs,Favipiravir 4tabs,Other Medications
0,1,Day 1,AM Time,132.0000,78.0000,97.0000,20.0000,36.5000,99.0000,3.0000,...,-,-,A,0.0000,2.0000,No,Y,NaN,NaN,NaN
1,1,Day 1,PM Time,125.0000,85.0000,75.0000,20.0000,36.6000,97.0000,3.0000,...,-,-,A,0.0000,2.0000,No,Y,NaN,NaN,NaN
2,1,Day 2,AM Time,143.0000,52.0000,92.0000,18.0000,36.5000,95.0000,3.0000,...,-,-,A,0.0000,2.0000,No,Y,NaN,NaN,NaN
3,1,Day 2,PM Time,141.0000,78.0000,79.0000,19.0000,36.8000,99.0000,3.0000,...,-,-,A,0.0000,2.0000,No,Y,NaN,NaN,NaN
4,1,Day 3,AM Time,137.0000,79.0000,88.0000,20.0000,36.5000,98.0000,3.0000,...,-,-,A,0.0000,2.0000,No,Y,NaN,NaN,NaN


In [206]:
df_merge_1.isna().count()

Study Num                   3355
Day                         3355
Time                        3355
SBP                         3355
DBP                         3355
HR                          3355
RR                          3355
Temp                        3355
Pulse Ox                    3355
Score Px Condition          3355
Score Coughing              3355
Score Sore Throat           3355
Score Headache              3355
Score Muscle Pain           3355
Score Colds                 3355
Score Chills or Sweating    3355
Score Malaise or Fatigue    3355
Score Chest Pain            3355
Score Dehydration           3355
Score Cyanosis              3355
Score Pleural Effusion      3355
Score Thoracic Rales        3355
Score Mental Status         3355
NEWS Score                  3355
STATUS Score                3355
Any ADVERSE EVENT           3355
Supportive Care             3355
Favipiravir 9tabs           3355
Favipiravir 4tabs           3355
Other Medications           3355
dtype: int

In [207]:
# df_merge_1.to_excel('test.xlsx', index=False)

In [208]:
## 2nd merge
## Key: Study_Num, Day,
df_chest_day1 = pd.read_excel('07. Chest Imaging - Day 1.xlsx')
df_chest_improv = pd.read_excel('07. Chest Imaging - improved or worse.xlsx')

In [221]:
df_chest_mrg = df_chest_day1.append(df_chest_improv, sort=False).sort_values(by=['Study Num'])
# df_chest_mrg.reset_index()
df_chest_mrg.head()

,Study Num,Day,Date of Test,Infiltrates,Describe,Pneumonia,Pulmonary Congestion,Pleura Effusion,Other findings,Improved,Worse
0,1,Day 1,2020-10-10,Y,suspicious alveolointersitial infiltrates left...,Y,N,N,"infiltrates over the left upper lobe, suggest ...",NaN,NaN
2,1,Day 10,2020-10-27,NaN,PNEUMONIA WITH SIGNS OF REGRESSION. Linear opa...,NaN,NaN,NaN,NaN,Y,N
1,1,Day 7,2020-10-23,NaN,IMPRESSION: â€¢ PNEUMONIA WITH SIGNS OF REGRES...,NaN,NaN,NaN,NaN,Y,N
0,1,Day 4,2020-10-20,NaN,GROUND GLASS OPACITIES ARE NOW NOTED IN BOTH L...,NaN,NaN,NaN,NaN,N,Y
7,2,Day 28,2020-11-18,NaN,Near complete clearing of the previously noted...,NaN,NaN,NaN,NaN,Y,N


In [222]:
# df_chest_mrg.to_excel('test_chest.xlsx', index=False)

In [227]:
# df_merge_2 = df_chest_mrg.append(df_lab_test, sort=False).sort_values(by=['Study Num'])
df_merge_2 = df_chest_mrg.merge(df_lab_test, how='outer', on=['Study Num', 'Day']).drop_duplicates()
len(df_merge_2)

739

In [228]:
df_merge_2.head()

,Study Num,Day,Date of Test,Infiltrates,Describe,Pneumonia,Pulmonary Congestion,Pleura Effusion,Other findings,Improved,...,Na,Cl,FBS,CRP,Procal,Ferritin,DDimer,Urine RBC,Urine Sugar,Urine Protein
0,1,Day 1,2020-10-10,Y,suspicious alveolointersitial infiltrates left...,Y,N,N,"infiltrates over the left upper lobe, suggest ...",NaN,...,137,99,90,192,0.1300,348,5.5700,260,0,0
1,1,Day 10,2020-10-27,NaN,PNEUMONIA WITH SIGNS OF REGRESSION. Linear opa...,NaN,NaN,NaN,NaN,Y,...,139,100,4.4000,48,0.0700,523,5,0,0,0
2,1,Day 7,2020-10-23,NaN,IMPRESSION: â€¢ PNEUMONIA WITH SIGNS OF REGRES...,NaN,NaN,NaN,NaN,Y,...,138,100,87,8.6400,0.0600,348,5.5000,260,0,0
3,1,Day 4,2020-10-20,NaN,GROUND GLASS OPACITIES ARE NOW NOTED IN BOTH L...,NaN,NaN,NaN,NaN,N,...,138,100,5,34.5700,0.0700,NaN,NaN,260,0,0
4,2,Day 28,2020-11-18,NaN,Near complete clearing of the previously noted...,NaN,NaN,NaN,NaN,Y,...,146.4000,110.6000,4.5400,1.7000,0.0700,137.0300,NaN,3,0,0


In [230]:
df_merge_3 = df_merge_2.merge(df_quant, how='outer', on=['Study Num', 'Day']).drop_duplicates()
df_merge_3.head()

,Study Num,Day,Date of Test_x,Infiltrates,Describe,Pneumonia,Pulmonary Congestion,Pleura Effusion,Other findings,Improved,...,CRP,Procal,Ferritin,DDimer,Urine RBC,Urine Sugar,Urine Protein,Date of Test_y,Supportive,Favipiravir
0,1,Day 1,2020-10-10,Y,suspicious alveolointersitial infiltrates left...,Y,N,N,"infiltrates over the left upper lobe, suggest ...",NaN,...,192,0.1300,348,5.5700,260,0,0,2020-10-17,SARS-CoV-2 (causative agent of COVID-19) Viral...,-
1,1,Day 10,2020-10-27,NaN,PNEUMONIA WITH SIGNS OF REGRESSION. Linear opa...,NaN,NaN,NaN,NaN,Y,...,48,0.0700,523,5,0,0,0,2020-10-26,0,-
2,1,Day 7,2020-10-23,NaN,IMPRESSION: â€¢ PNEUMONIA WITH SIGNS OF REGRES...,NaN,NaN,NaN,NaN,Y,...,8.6400,0.0600,348,5.5000,260,0,0,2020-10-23,0,-
3,1,Day 4,2020-10-20,NaN,GROUND GLASS OPACITIES ARE NOW NOTED IN BOTH L...,NaN,NaN,NaN,NaN,N,...,34.5700,0.0700,NaN,NaN,260,0,0,2020-10-20,0,-
4,2,Day 28,2020-11-18,NaN,Near complete clearing of the previously noted...,NaN,NaN,NaN,NaN,Y,...,1.7000,0.0700,137.0300,NaN,3,0,0,2020-11-18,0,NaN


In [233]:
df_merge_4 = df_merge_3.merge(df_ekg, how='outer', on=['Study Num', 'Day']).drop_duplicates()
df_merge_4.head()

,Study Num,Day,Date of Test_x,Infiltrates,Describe,Pneumonia,Pulmonary Congestion,Pleura Effusion,Other findings,Improved,...,Supportive,Favipiravir,Date of Test,HR,PR,QRS,QT,Other abnormalities,Specify,Clinical Significance
0,1,Day 1,2020-10-10,Y,suspicious alveolointersitial infiltrates left...,Y,N,N,"infiltrates over the left upper lobe, suggest ...",NaN,...,SARS-CoV-2 (causative agent of COVID-19) Viral...,-,2020-10-17,83.0000,0.1200,0.0400,0.3600,N,NaN,Normal
1,1,Day 10,2020-10-27,NaN,PNEUMONIA WITH SIGNS OF REGRESSION. Linear opa...,NaN,NaN,NaN,NaN,Y,...,0,-,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Day 7,2020-10-23,NaN,IMPRESSION: â€¢ PNEUMONIA WITH SIGNS OF REGRES...,NaN,NaN,NaN,NaN,Y,...,0,-,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Day 4,2020-10-20,NaN,GROUND GLASS OPACITIES ARE NOW NOTED IN BOTH L...,NaN,NaN,NaN,NaN,N,...,0,-,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,Day 28,2020-11-18,NaN,Near complete clearing of the previously noted...,NaN,NaN,NaN,NaN,Y,...,0,NaN,2021-11-08,71.0000,186.0000,84.0000,396.0000,N,NaN,Normal


In [235]:
# df_merge_4.to_excel('test_merge.xlsx', index=False)

In [242]:
## 3rd merge
## Key: Study_Num
df_merge_last_1 = df_px_info.merge(df_admission, how='outer', on=['Study Num']).drop_duplicates()
len(df_merge_last_1)

155

In [241]:
df_merge_last_1.head(10)

,Study Num,Hospital Number,Last Name,First Name,Middle Name,Suffix,Sex,Birthdate,Date of Enrollment,Site,...,Severe Liver Impairment? (ALT>10x),"Has received other drugs CQ, HCQ, Lpv/Rit?",Severe Renal Impairment needing HD?,History of Gout or hyperuricemia?,Mental status change?,Has receiver favipiravir?,Pregnant?,Unable to consent to use condom for study +7 days,Female unable to consent contraception during study +7 days?,Male whose partner unable to consent contraception for study +7days?
0,1,4768849,A,M,R,NaN,Male,1979-03-31,2020-10-17,UP - PHILIPPINE GENERAL HOSPITAL,...,N,N,N,N,N,N,N,N,N,N
1,2,39509,R,A,D,JR,Male,1977-08-11,2020-10-22,DR. JOSE N. RODRIGUEZ MEMORIAL HOSPITAL,...,N,N,N,N,N,N,N,N,N,N
2,3,3579613,J,K,T,NOTAP,Female,1978-07-05,2020-10-31,UP - PHILIPPINE GENERAL HOSPITAL,...,N,N,N,N,N,N,N,N,N,N
3,4,1271950,O,W,A,NOTAP,Male,1959-10-01,2020-11-04,QUIRINO MEMORIAL MEDICAL CENTER,...,N,N,N,N,N,N,N,N,N,N
4,5,1271953,O,Y,G,NaN,Female,1953-02-17,2020-11-06,QUIRINO MEMORIAL MEDICAL CENTER,...,N,N,N,N,N,N,N,N,N,N
5,6,892841,V,F,Z,NaN,Male,1979-04-30,2020-11-19,DR. JOSE N. RODRIGUEZ MEMORIAL HOSPITAL,...,N,N,N,N,N,N,N,N,N,N
6,8,3495297,C,N,T,NaN,Female,1966-01-11,2020-11-21,UP - PHILIPPINE GENERAL HOSPITAL,...,N,N,N,N,N,N,N,N,N,N
7,9,893063,P,R,P,NaN,Male,1949-04-12,2020-11-24,DR. JOSE N. RODRIGUEZ MEMORIAL HOSPITAL,...,N,N,N,N,N,N,N,N,N,N
8,10,893377,C,J,Y,NaN,Female,1968-08-19,2020-11-28,DR. JOSE N. RODRIGUEZ MEMORIAL HOSPITAL,...,N,N,N,N,N,N,N,N,N,N
9,11,1337340,M,J,V,NOTAP,Female,1988-01-07,2020-11-29,QUIRINO MEMORIAL MEDICAL CENTER,...,N,N,N,N,N,N,NaN,N,N,N


In [247]:
df_merge_last_2 = df_merge_last_1.merge(df_end_trial, how='outer', on=['Study Num']).drop_duplicates()
len(df_merge_last_2)

155

In [255]:
df_merge_last_3 = df_merge_last_2.merge(df_adverse, how='outer', on=['Study Num']).drop_duplicates()
len(df_merge_last_3)

158

In [256]:
df_merge_last_4 = df_merge_last_3.merge(df_concomitant, how='outer', on=['Study Num']).drop_duplicates()
len(df_merge_last_4)

671

In [258]:
df_merge_last_4.to_excel('test_merge.xlsx', index=False)